# 方向词 Seed Expansion：硬约束超平面与四维审核输出

本 notebook 使用 2011Q1—2025Q4 的货币政策执行报告与货币政策委员会例会文本。流程为：语料去重与清洗 → jieba 切词 → 与金融 embedding 取交集 → 从《政策——方向.xlsx》的方向 seed 训练线性硬边界 → 对候选词输出正/非正、负/非负四维得分和上下文证据。

硬边界的含义：只有当线性 SVM 能把全部 seed 正确分类且函数间隔同号时才继续扩展；否则立即报错，不会在 seed 自身分错的情况下生成候选。四维概率是审核排序分数，不应解释为统计意义上的真实概率。

In [ ]:
from pathlib import Path
import sys, re, math, hashlib, warnings
from collections import Counter

ROOT = Path('/Users/xiaohan/Desktop/project/textmining')
LOCAL_PACKAGES = ROOT / '.seed_expansion_packages'
if LOCAL_PACKAGES.exists():
    sys.path.insert(0, str(LOCAL_PACKAGES))

import numpy as np
import pandas as pd
import jieba
from sklearn.svm import SVC

SEED_BOOK = ROOT / '政策——方向.xlsx'
EMBEDDING_PATH = ROOT / 'sgns.financial.word'
REPORT_DIR = ROOT / '央行沟通交流文本数据（2001-2026）' / '货币政策执行报告TXT'
MEETING_DIR = ROOT / 'monetary_policy_meetings' / 'raw_txt'
OUTPUT_XLSX = ROOT / '方向词_embedding扩展_人工审核.xlsx'
START_YEAR, END_YEAR = 2011, 2025
MIN_FREQ = 5
TOP_PER_SIDE = 100
MIN_SEED_COS = 0.18
RANDOM_STATE = 42
print('项目目录:', ROOT)
print('输出文件:', OUTPUT_XLSX)

## 1. 读取并按季度去重语料

执行报告按中文年份/季度识别；例会按 `YYYY_Qn` 识别。同一类语料同一季度若有多个文件，优先保留清洗后正文更长者。

In [ ]:
CN_Q = {'第一':1, '第二':2, '第三':3, '第四':4}
PAGE_JUNK = re.compile(r'=+\s*第?\s*\d+\s*页\s*=+|第\s*\d+\s*页')
URL = re.compile(r'https?://\S+|www\.\S+')
TABLE_NOISE = re.compile(r'(?:\d+[.、]){3,}|(?:\d{4}[.年/-]?){3,}|[A-Za-z]?\d+(?:\.\d+)?%?(?:\s+[A-Za-z]?\d+(?:\.\d+)?%?){5,}')
SPACE = re.compile(r'\s+')

def read_text(path):
    for enc in ('utf-8', 'utf-8-sig', 'gb18030'):
        try:
            return path.read_text(encoding=enc)
        except UnicodeDecodeError:
            pass
    return path.read_text(encoding='utf-8', errors='ignore')

def clean_text(text):
    text = PAGE_JUNK.sub('。', text)
    text = URL.sub(' ', text)
    text = TABLE_NOISE.sub(' ', text)
    text = re.sub(r'[\u200b\ufeff\xa0]', ' ', text)
    text = SPACE.sub(' ', text)
    return text.strip()

def period_from_name(path, kind):
    name = path.name
    if kind == '例会':
        m = re.search(r'(20\d{2})_Q([1-4])', name)
        return (int(m.group(1)), int(m.group(2))) if m else None
    m = re.search(r'(20\d{2})年(第一|第二|第三|第四)季度', name)
    return (int(m.group(1)), CN_Q[m.group(2)]) if m else None

def collect_quarterly_files(folder, kind):
    grouped = {}
    for p in folder.glob('*.txt'):
        period = period_from_name(p, kind)
        if not period or not (START_YEAR <= period[0] <= END_YEAR):
            continue
        text = clean_text(read_text(p))
        row = {'kind': kind, 'year': period[0], 'quarter': period[1], 'path': p, 'text': text}
        if period not in grouped or len(text) > len(grouped[period]['text']):
            grouped[period] = row
    return list(grouped.values())

docs = collect_quarterly_files(REPORT_DIR, '执行报告') + collect_quarterly_files(MEETING_DIR, '例会')
docs = sorted(docs, key=lambda x: (x['year'], x['quarter'], x['kind']))
manifest = pd.DataFrame([{k: (str(v) if k == 'path' else v) for k, v in d.items() if k != 'text'} | {'字符数':len(d['text'])} for d in docs])
print(manifest.groupby('kind').size())
print('总文件数:', len(docs), '覆盖季度:', manifest[['year','quarter']].drop_duplicates().shape[0])

## 2. 读取方向 seed，清洗、切词并构造语料词频

标准词与别名都会作为 seed；候选词过滤纯数字、单字、停用/版面词、英文碎片以及极长词。保留原句上下文用于人工识别伪相关。

In [ ]:
seed_df = pd.read_excel(SEED_BOOK, sheet_name='02方向词')
seed_df = seed_df[seed_df['方向标签'].isin([1, -1])].copy()
seed_label = {}
seed_source = {}
for _, r in seed_df.iterrows():
    terms = [str(r['方向词标准名']).strip()]
    if pd.notna(r.get('别名/变体')):
        terms += [x.strip() for x in re.split(r'[；;、,/]', str(r['别名/变体'])) if x.strip()]
    for t in terms:
        old = seed_label.get(t)
        if old is not None and old != int(r['方向标签']):
            raise ValueError(f'方向 seed 标签冲突: {t}')
        seed_label[t] = int(r['方向标签'])
        seed_source[t] = str(r['方向词标准名']).strip()
        jieba.add_word(t, freq=1000000)

STOPWORDS = set('的 了 和 是 在 对 将 与 及 为 等 中 上 下 也 有 被 从 到 由 更 已 并 或 这 其 该 我们 目前 其中 一个 一种 以及 进行 通过 相关 方面 情况 问题 工作 进一步 不断 继续 同时 主要 重要 有关 充分 切实 积极 稳妥 按照 坚持 加强 做好 推动 促进 完善 提升 实现 保持 发挥 支持'.split())
STOPWORDS |= {'中国','人民银行','央行','报告','季度','会议','委员会','货币政策','金融','经济','政策','市场','银行','我国','全国','今年','去年','同比','环比','亿元','百分点','数据','图表','资料来源'}
VALID_TOKEN = re.compile(r'^[\u4e00-\u9fff]{2,8}$')
sentence_splitter = re.compile(r'[。！？!?；;\n]+')
freq = Counter()
context = {}
for d in docs:
    for sent in sentence_splitter.split(d['text']):
        sent = sent.strip()
        if len(sent) < 4:
            continue
        for token in jieba.cut(sent, HMM=False):
            token = token.strip()
            if not VALID_TOKEN.fullmatch(token) or token in STOPWORDS:
                continue
            freq[token] += 1
            if token not in context:
                context[token] = {
                    '原文例句': sent[:240], '来源文件': d['path'].name,
                    '年份': d['year'], '季度': d['quarter'], '语料类型': d['kind']
                }
candidate_vocab = {w for w, n in freq.items() if n >= MIN_FREQ}
wanted_vocab = candidate_vocab | set(seed_label)
print('方向 seed（含别名）:', len(seed_label), '语料候选:', len(candidate_vocab))

## 3. 与 embedding 词典取交集

模型约 1.2GB，因此逐行扫描，只保留 seed 和语料候选所需的向量，不把整个 embedding 常驻内存。

In [ ]:
vectors = {}
with EMBEDDING_PATH.open('r', encoding='utf-8', errors='ignore') as f:
    header = f.readline().split()
    embedding_rows, embedding_dim = int(header[0]), int(header[1])
    for line in f:
        word, sep, rest = line.partition(' ')
        if word not in wanted_vocab:
            continue
        arr = np.fromstring(rest, sep=' ', dtype=np.float32)
        if arr.size == embedding_dim and np.isfinite(arr).all():
            norm = np.linalg.norm(arr)
            if norm > 0:
                vectors[word] = arr / norm
print('embedding:', embedding_rows, 'x', embedding_dim)
print('语料∩embedding:', len(candidate_vocab & vectors.keys()), '/', len(candidate_vocab))
missing_seeds = sorted(set(seed_label) - vectors.keys())
print('缺失 seed:', missing_seeds)

## 4. 训练必须完整切割 seed 的线性超平面

逐级增大 `C` 逼近硬间隔，并显式检查全部 seed 的预测标签和带符号函数间隔。若任何 seed 位于错误侧或边界上，停止运行。

In [ ]:
seed_terms = sorted(set(seed_label) & vectors.keys())
X_seed = np.vstack([vectors[w] for w in seed_terms])
y_seed = np.array([seed_label[w] for w in seed_terms], dtype=int)
if set(y_seed) != {-1, 1}:
    raise RuntimeError('embedding 中至少需要一组正 seed 和一组负 seed。')

hard_svm = None
for C in (1e2, 1e3, 1e4, 1e5, 1e6, 1e7):
    model = SVC(kernel='linear', C=C, class_weight=None, random_state=RANDOM_STATE)
    model.fit(X_seed, y_seed)
    margin = y_seed * model.decision_function(X_seed)
    if np.all(model.predict(X_seed) == y_seed) and margin.min() > 1e-8:
        hard_svm = model
        break
if hard_svm is None:
    raise RuntimeError('现有 seed 在该 embedding 中不能被线性超平面完整切割；请审核冲突 seed，程序不会带错扩展。')

seed_decision = hard_svm.decision_function(X_seed)
seed_margin = y_seed * seed_decision
print('采用 C =', hard_svm.C)
print('seed 完整切割:', bool(np.all(hard_svm.predict(X_seed) == y_seed)))
print('最小/中位带符号函数间隔:', float(seed_margin.min()), float(np.median(seed_margin)))

## 5. 四维评分、双侧扩展与证据抽取

用 seed 中位函数间隔作为温度缩放：`P(正)=sigmoid(score/T)`、`P(非正)=1-P(正)`、`P(负)=sigmoid(-score/T)`、`P(非负)=1-P(负)`。这四列用于两个二元审核问题，数学上存在互补关系。候选综合排序同时考虑超平面置信度、与同侧 seed 最大余弦相似度及语料频次。

In [ ]:
def sigmoid(x):
    x = np.clip(x, -30, 30)
    return 1.0 / (1.0 + np.exp(-x))

pos_seed_matrix = X_seed[y_seed == 1]
neg_seed_matrix = X_seed[y_seed == -1]
temperature = max(float(np.median(seed_margin)), 0.05)
rows = []
for word in sorted(candidate_vocab & vectors.keys()):
    if word in seed_label:
        continue
    v = vectors[word]
    raw_score = float(hard_svm.decision_function(v.reshape(1, -1))[0])
    p_pos = float(sigmoid(raw_score / temperature))
    p_neg = float(sigmoid(-raw_score / temperature))
    cos_pos = float(np.max(pos_seed_matrix @ v))
    cos_neg = float(np.max(neg_seed_matrix @ v))
    side = 1 if raw_score > 0 else -1
    side_cos = cos_pos if side == 1 else cos_neg
    if side_cos < MIN_SEED_COS:
        continue
    confidence = abs(p_pos - 0.5) * 2
    freq_score = math.log1p(freq[word]) / math.log1p(max(freq.values()))
    rank_score = 0.55 * confidence + 0.30 * max(side_cos, 0) + 0.15 * freq_score
    c = context.get(word, {})
    rows.append({
        '候选词': word, '模型侧': '正向候选' if side == 1 else '负向候选',
        '建议方向标签': side, 'P(正)': p_pos, 'P(非正)': 1-p_pos,
        'P(负)': p_neg, 'P(非负)': 1-p_neg, '超平面分数': raw_score,
        '正seed最大余弦': cos_pos, '负seed最大余弦': cos_neg,
        '综合排序分': rank_score, '语料词频': freq[word], **c,
        '人工审核': '待审核', '人工标签': '', '审核备注': ''
    })
scored = pd.DataFrame(rows)
selected = (scored.sort_values(['模型侧','综合排序分','语料词频'], ascending=[True,False,False])
            .groupby('模型侧', group_keys=False).head(TOP_PER_SIDE)
            .sort_values(['建议方向标签','综合排序分'], ascending=[False,False]).reset_index(drop=True))
selected.insert(0, '候选ID', [f'DIR-{i:04d}' for i in range(1, len(selected)+1)])
print(selected.groupby('模型侧').size())
selected.head(10)

## 6. 导出独立人工审核工作簿

工作簿包含：候选审核、seed 硬边界检查、语料/模型覆盖、参数说明。人工审核列不会自动进入 seed；审核后应另存版本，再显式合并。

In [ ]:
seed_audit = pd.DataFrame({
    'seed词': seed_terms, '标准词': [seed_source[w] for w in seed_terms],
    '既有标签': y_seed, '模型预测': hard_svm.predict(X_seed),
    '超平面分数': seed_decision, '带符号函数间隔': seed_margin,
    '硬边界通过': hard_svm.predict(X_seed) == y_seed
})
coverage = manifest.copy()
coverage['path'] = coverage['path'].astype(str)
params = pd.DataFrame([
    ['样本期', f'{START_YEAR}Q1—{END_YEAR}Q4'],
    ['切词方法', 'jieba精确模式，HMM=False；seed强制加入用户词典'],
    ['最低语料词频', MIN_FREQ], ['每侧输出数', TOP_PER_SIDE],
    ['最低同侧seed余弦', MIN_SEED_COS], ['SVM核', 'linear'],
    ['硬边界C', hard_svm.C], ['seed最小带符号函数间隔', float(seed_margin.min())],
    ['embedding', EMBEDDING_PATH.name], ['embedding维度', embedding_dim],
    ['四维解释', 'P(正)/P(非正)与P(负)/P(非负)为两组互补审核分数，并非真实概率'],
    ['纳入规则', '进入语料∩embedding、达到最低频次/同侧相似度、按综合分每侧取Top N'],
    ['人工规则', '只有人工审核为保留且人工标签确认后，才能并入正式seed'],
], columns=['参数','取值'])

with pd.ExcelWriter(OUTPUT_XLSX, engine='xlsxwriter') as writer:
    selected.to_excel(writer, sheet_name='01候选审核', index=False)
    seed_audit.to_excel(writer, sheet_name='02Seed硬边界', index=False)
    coverage.to_excel(writer, sheet_name='03语料覆盖', index=False)
    params.to_excel(writer, sheet_name='00方法与参数', index=False)
    wb = writer.book
    header_fmt = wb.add_format({'bold':True,'font_color':'#FFFFFF','bg_color':'#1F4E78','align':'center','valign':'vcenter','border':0})
    pos_fmt = wb.add_format({'bg_color':'#E2F0D9'})
    neg_fmt = wb.add_format({'bg_color':'#FCE4D6'})
    pct_fmt = wb.add_format({'num_format':'0.000'})
    int_fmt = wb.add_format({'num_format':'0'})
    for sheet_name, frame in [('01候选审核',selected),('02Seed硬边界',seed_audit),('03语料覆盖',coverage),('00方法与参数',params)]:
        ws = writer.sheets[sheet_name]
        ws.freeze_panes(1, 1)
        ws.autofilter(0, 0, len(frame), len(frame.columns)-1)
        ws.set_row(0, 26, header_fmt)
        for j, col in enumerate(frame.columns):
            width = min(max(len(str(col))+2, int(frame[col].astype(str).str.len().quantile(.90))+2 if len(frame) else 10), 48)
            ws.set_column(j, j, width)
    ws = writer.sheets['01候选审核']
    cols = {c:i for i,c in enumerate(selected.columns)}
    for c in ['P(正)','P(非正)','P(负)','P(非负)','超平面分数','正seed最大余弦','负seed最大余弦','综合排序分']:
        ws.set_column(cols[c], cols[c], 14, pct_fmt)
    ws.set_column(cols['语料词频'], cols['语料词频'], 11, int_fmt)
    ws.conditional_format(1, cols['建议方向标签'], len(selected), cols['建议方向标签'], {'type':'cell','criteria':'==','value':1,'format':pos_fmt})
    ws.conditional_format(1, cols['建议方向标签'], len(selected), cols['建议方向标签'], {'type':'cell','criteria':'==','value':-1,'format':neg_fmt})
    ws.data_validation(1, cols['人工审核'], max(len(selected),1), cols['人工审核'], {'validate':'list','source':['待审核','保留','删除','待定']})
    ws.data_validation(1, cols['人工标签'], max(len(selected),1), cols['人工标签'], {'validate':'list','source':['','1','-1','0']})
print('已导出:', OUTPUT_XLSX, '候选数:', len(selected))